In [1]:
import os
from dotenv import load_dotenv
from src.ingest import ingest_documents
from src.retriever import get_collection, retrieve
from src.rag_pipeline import rag_answer
from src.structured import generate_structured_output
import pandas as pd

load_dotenv()
print("Ready.")

Ready.


In [2]:
collection = ingest_documents('data/pdfs/')
print(f"Collection ready with {collection.count()} chunks")

Loaded: CE308_ProjectReport.pdf - 15896 characters
Loaded: ProjectReport_2023037_2023694.pdf - 19542 characters
Loaded: Raid Storage Study Guide.pdf - 8513 characters
CE308_ProjectReport.pdf: 36 chunks
ProjectReport_2023037_2023694.pdf: 44 chunks
Raid Storage Study Guide.pdf: 19 chunks
Stored 99 chunks in ChromaDB
Collection ready with 99 chunks


In [3]:
collection = get_collection()
query = "What is the main topic of the document?"
chunks, sources = retrieve(query, collection)

print(f"Query: {query}\n")
for i, (chunk, source) in enumerate(zip(chunks, sources)):
    print(f"--- Chunk {i+1} from {source} ---")
    print(chunk[:300])
    print()

Query: What is the main topic of the document?

--- Chunk 1 from ProjectReport_2023037_2023694.pdf ---
pirational References 
References were drawn from data journalism, print, UI design, and books to ensure a 
diverse design vocabulary: 
Source Medium Design Element Borrowed 
NYT ‘The Air We 
Breathe’ (2023) 
Web / Data 
Journalism 
Dark background with AQI color scale; 
scrollytelling pacing and an

--- Chunk 2 from ProjectReport_2023037_2023694.pdf ---
el 
• Progress bar: Fixed at top — shows scroll position through the full story (0–
100%) 
• 8 nav dots: Fixed on right side — click any dot to jump directly to that scene 
• Scenes 1–3 and 8: Author-driven — content appears and animates on scroll 
• Scenes 4–6: Hybrid — charts animate on scroll-ent

--- Chunk 3 from ProjectReport_2023037_2023694.pdf ---
ts 
The Pudding — 
Visual Essays Web Full-viewport sections; smooth transitions; 
reader engagement hooks 
Bloomberg CityLab 
Smog Coverage 
Web / 
Newsprint 
Annotated line charts; n

In [4]:
questions = [
    "What is the main topic of the document?",
    "Summarize the key points.",
    "What conclusions are drawn?"
]

for question in questions:
    answer, sources, chunks = rag_answer(question, collection)
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f"Sources: {set(sources)}")
    print()

Question: What is the main topic of the document?
Answer: The document is a set of inspirational references and interaction/UI specifications for a scrollytelling web data story about air quality (AQI/smog), drawing from sources such as NYT’s “The Air We Breathe,” IQAir World Air Quality Report, Bloomberg CityLab Smog Coverage, NASA Earth Observatory, The Pudding, Tufte, and Figma UI kits.
Sources: {'ProjectReport_2023037_2023694.pdf'}

Question: Summarize the key points.
Answer: Based on the provided context, here is a summary of the key points:

*   **Project Overview**: The text is from the "GIKI Data Visualization Final Project Report" (2026) detailing an "Air Quality Narrative Visualization."
*   **Data Visualizations & Scenes**:
    *   **Correlation Scene (Scene 5 context)**: Uses a bar chart to show relationships between AQI and variables like Surface Pressure, Precipitation, and Wind Speed. Bars are annotated with *r* values, animate from the center outward on scroll, and use 

In [5]:
movies = [
    "The Dark Knight - a superhero film about Batman fighting the Joker in Gotham City",
    "Titanic - a romance film about two people falling in love on a doomed ship",
    "Interstellar - a sci-fi film about astronauts travelling through a wormhole to save humanity"
]

results = []
for movie in movies:
    validated, raw, error = generate_structured_output(movie)
    if validated:
        print(f"✓ Valid output for: {validated.title}")
        print(f"  Genre: {validated.genre}")
        print(f"  Rating: {validated.rating}")
        print(f"  Recommended: {validated.recommended}")
        results.append(validated.model_dump())
    else:
        print(f"✗ Validation failed: {error}")
        print(f"  Raw output: {raw}")
    print()

✓ Valid output for: The Dark Knight
  Genre: Superhero/Crime Thriller
  Rating: 9.0
  Recommended: True

✓ Valid output for: Titanic
  Genre: Romance
  Rating: 7.8
  Recommended: True

✓ Valid output for: Interstellar
  Genre: Science Fiction
  Rating: 8.6
  Recommended: True



In [6]:
if results:
    df = pd.DataFrame(results)
    df

In [7]:
print("""
Hallucination Analysis
======================
RAG systems can still hallucinate in these ways:

1. Retrieved chunks are irrelevant - model makes up an answer
   Detection: check if answer contains words from retrieved chunks

2. Model ignores context and uses its own knowledge
   Detection: ask about something NOT in the documents and see if it says I do not know

3. Structured output has wrong values even if schema is valid
   Detection: rating of 15.0 passes schema (float) but is out of range 0-10
   Fix: add validator in Pydantic using field_validator

Example where hallucination was caught:
- Asked about a topic not in the PDFs
- Model answered confidently instead of saying I do not know
- Fix: added explicit instruction in prompt to say I do not know if not in context
""")


Hallucination Analysis
RAG systems can still hallucinate in these ways:

1. Retrieved chunks are irrelevant - model makes up an answer
   Detection: check if answer contains words from retrieved chunks

2. Model ignores context and uses its own knowledge
   Detection: ask about something NOT in the documents and see if it says I do not know

3. Structured output has wrong values even if schema is valid
   Detection: rating of 15.0 passes schema (float) but is out of range 0-10
   Fix: add validator in Pydantic using field_validator

Example where hallucination was caught:
- Asked about a topic not in the PDFs
- Model answered confidently instead of saying I do not know
- Fix: added explicit instruction in prompt to say I do not know if not in context



## Week 3 Findings

### RAG Pipeline
Built a RAG system that loads PDFs, chunks them into 500 character
overlapping segments, stores them in ChromaDB, and retrieves the
most relevant chunks for any query. The LLM then answers using
only the retrieved context.

### Chunking Strategy
Used fixed size chunks of 500 characters with 50 character overlap.
Overlap ensures that sentences split across chunk boundaries are
still captured in at least one chunk.

### Embedding Comparison
ChromaDB uses its default embedding model internally. The retrieval
quality depends on how semantically similar the query is to the chunks.

### Structured Outputs
Built a pipeline where the LLM returns JSON validated against a
Pydantic schema. If the output is missing a field or has the wrong
type the validation fails and the error is caught and logged.

### Hallucination Detection
When the model was asked about topics not in the documents it
sometimes answered confidently instead of saying I do not know.
Fixed by adding explicit instructions in the prompt. Schema
validation catches type errors but not value errors like a
rating of 15 which is a valid float but out of range.